
# Consumption–Saving RL — PPO with Normalized Action Space

이 노트북은 이전 PPO 실패의 가장 유력한 원인인 **비대칭 action space [0, 0.95]**를 수정합니다.

Stable-Baselines3 PPO는 continuous Box action에서 Gaussian policy를 사용하므로,
환경에 제공하는 RL action을 **[-1, 1]**로 정규화합니다.

경제적 saving share는 다음과 같이 변환합니다.

\[
u_t \in [-1,1]
\]

\[
z_t = 0.475(u_t + 1)
\]

따라서:
- \(u=-1 \Rightarrow z=0\)
- \(u=0 \Rightarrow z=0.475\)
- \(u=1 \Rightarrow z=0.95\)

경제모형 자체는 전혀 바뀌지 않습니다.
오직 PPO가 보는 action 표현만 바뀝니다.

이번 단계는 seed=42, 50k timesteps의 smoke test입니다.


In [ ]:

# ============================================================
# 0. Runtime setup
# ============================================================

import sys
import subprocess
import importlib

try:
    import gymnasium as gym
except ModuleNotFoundError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "gymnasium"]
    )
    import gymnasium as gym

import sympy
sympy_printing = importlib.import_module("sympy.printing")
if not hasattr(sympy, "printing"):
    sympy.printing = sympy_printing

try:
    import stable_baselines3
except ModuleNotFoundError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "stable-baselines3"]
    )
    import stable_baselines3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from gymnasium import spaces

import torch

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor

print("Python:", sys.version.split()[0])
print("SymPy:", sympy.__version__)
print("PyTorch:", torch.__version__)
print("Gymnasium:", gym.__version__)
print("Stable-Baselines3:", stable_baselines3.__version__)


In [ ]:

# ============================================================
# 1. Economic calibration
# ============================================================

R = 1.00985
BETA = 0.9703
SIGMA = 2.0

Y_HIGH = 1.0
Y_LOW = 0.472

P_INCOME = np.array(
    [
        [0.9393, 0.0607],
        [0.3920, 0.6080]
    ],
    dtype=np.float64
)

A_MAX_DP = 4.5

def action_to_saving_share(action):
    """
    RL action u in [-1, 1]
        -> economic saving share z in [0, 0.95]
    """
    u = np.clip(
        np.asarray(action, dtype=np.float64),
        -1.0,
        1.0
    )

    return 0.475 * (u + 1.0)


print("Calibration loaded.")
print("u=-1 -> z =", action_to_saving_share([-1.0])[0])
print("u= 0 -> z =", action_to_saving_share([0.0])[0])
print("u= 1 -> z =", action_to_saving_share([1.0])[0])


In [ ]:

# ============================================================
# 2. PPO environment with normalized action [-1, 1]
# ============================================================

class NormalizedContinuousConsumptionSavingEnv(gym.Env):

    metadata = {"render_modes": []}

    def __init__(
        self,
        random_initial_state=True,
        initial_asset=1.0,
        initial_income_state=0,
        R=R,
        beta=BETA,
        sigma=SIGMA,
        y_high=Y_HIGH,
        y_low=Y_LOW,
        transition_matrix=P_INCOME
    ):
        super().__init__()

        self.R = float(R)
        self.beta = float(beta)
        self.sigma = float(sigma)

        self.income_values = np.array(
            [y_high, y_low],
            dtype=np.float64
        )

        self.P = np.asarray(
            transition_matrix,
            dtype=np.float64
        )

        self.random_initial_state = bool(random_initial_state)

        self.initial_asset = float(initial_asset)
        self.initial_income_state = int(initial_income_state)

        # IMPORTANT:
        # normalized symmetric action space for PPO
        self.action_space = spaces.Box(
            low=np.array([-1.0], dtype=np.float32),
            high=np.array([1.0], dtype=np.float32),
            dtype=np.float32
        )

        # observation = [asset / 4.5, income]
        self.observation_space = spaces.Box(
            low=np.array([0.0, Y_LOW], dtype=np.float32),
            high=np.array([np.inf, Y_HIGH], dtype=np.float32),
            dtype=np.float32
        )

        self.asset = None
        self.income_state = None

    def _obs(self):
        return np.array(
            [
                self.asset / 4.5,
                self.income_values[self.income_state]
            ],
            dtype=np.float32
        )

    def _utility(self, consumption):

        if np.isclose(self.sigma, 1.0):
            return float(np.log(consumption))

        return float(
            consumption ** (1.0 - self.sigma)
            / (1.0 - self.sigma)
        )

    def reset(self, *, seed=None, options=None):

        super().reset(seed=seed)

        options = options or {}

        if "initial_asset" in options:
            self.asset = float(options["initial_asset"])
        elif self.random_initial_state:
            self.asset = float(
                self.np_random.uniform(0.0, 4.5)
            )
        else:
            self.asset = self.initial_asset

        if "initial_income_state" in options:
            self.income_state = int(
                options["initial_income_state"]
            )
        elif self.random_initial_state:
            self.income_state = int(
                self.np_random.integers(0, 2)
            )
        else:
            self.income_state = self.initial_income_state

        return self._obs(), {}

    def step(self, action):

        # PPO action [-1,1] -> economic saving share [0,.95]
        z = float(
            action_to_saving_share(action).reshape(-1)[0]
        )

        income = self.income_values[
            self.income_state
        ]

        cash_on_hand = (
            self.R * self.asset
            + income
        )

        next_asset = (
            z * cash_on_hand
        )

        consumption = (
            (1.0 - z)
            * cash_on_hand
        )

        reward = self._utility(
            consumption
        )

        next_income_state = int(
            self.np_random.choice(
                2,
                p=self.P[self.income_state]
            )
        )

        self.asset = next_asset
        self.income_state = next_income_state

        terminated = False
        truncated = False

        return (
            self._obs(),
            reward,
            terminated,
            truncated,
            {
                "saving_share": z
            }
        )


def make_training_env():

    env = NormalizedContinuousConsumptionSavingEnv(
        random_initial_state=True
    )

    # computational truncation only
    env = gym.wrappers.TimeLimit(
        env,
        max_episode_steps=200
    )

    env = Monitor(env)

    return env


print("Normalized PPO environment ready.")


In [ ]:

# ============================================================
# 3. DP-Fine benchmark
# ============================================================

def solve_dp_fine(
    R=R,
    beta=BETA,
    sigma=SIGMA,
    y_high=Y_HIGH,
    y_low=Y_LOW,
    transition_matrix=P_INCOME,
    a_min=0.0,
    a_max=A_MAX_DP,
    n_assets=500,
    tolerance=1e-10,
    max_iterations=5000,
):

    P = np.asarray(
        transition_matrix,
        dtype=np.float64
    )

    income_values = np.array(
        [y_high, y_low],
        dtype=np.float64
    )

    asset_grid = np.linspace(
        a_min,
        a_max,
        n_assets
    )

    V = np.zeros(
        (n_assets, 2),
        dtype=np.float64
    )

    policy_asset = np.zeros_like(V)

    for iteration in range(max_iterations):

        V_new = np.empty_like(V)

        for income_state in range(2):

            income = income_values[income_state]

            cash_on_hand = (
                R * asset_grid
                + income
            )

            consumption = (
                cash_on_hand[:, None]
                - asset_grid[None, :]
            )

            utility = np.full(
                consumption.shape,
                -np.inf,
                dtype=np.float64
            )

            feasible = consumption > 1e-12

            if np.isclose(sigma, 1.0):
                utility[feasible] = np.log(
                    consumption[feasible]
                )
            else:
                utility[feasible] = (
                    consumption[feasible]
                    ** (1.0 - sigma)
                    / (1.0 - sigma)
                )

            expected_future = (
                P[income_state, 0] * V[:, 0]
                +
                P[income_state, 1] * V[:, 1]
            )

            Q = (
                utility
                +
                beta * expected_future[None, :]
            )

            best_idx = np.argmax(
                Q,
                axis=1
            )

            V_new[:, income_state] = (
                Q[
                    np.arange(n_assets),
                    best_idx
                ]
            )

            policy_asset[:, income_state] = (
                asset_grid[best_idx]
            )

        diff = np.max(
            np.abs(V_new - V)
        )

        V = V_new

        if diff < tolerance:
            print(
                "DP-Fine converged after",
                iteration + 1,
                "iterations."
            )
            break

    return asset_grid, V, policy_asset


asset_grid_fine, V_dp_fine, policy_asset_fine = (
    solve_dp_fine()
)

dp_fine_value = np.interp(
    1.0,
    asset_grid_fine,
    V_dp_fine[:, 0]
)

cash_high = (
    R * asset_grid_fine + Y_HIGH
)

cash_low = (
    R * asset_grid_fine + Y_LOW
)

policy_share_fine = np.column_stack(
    [
        policy_asset_fine[:, 0] / cash_high,
        policy_asset_fine[:, 1] / cash_low
    ]
)

print(
    "DP-Fine Bellman value at (asset=1, High):",
    dp_fine_value
)


In [ ]:

# ============================================================
# 4. Evaluation helpers
# ============================================================

def utility_vectorized(
    consumption,
    sigma=SIGMA
):
    if np.isclose(sigma, 1.0):
        return np.log(consumption)

    return (
        consumption ** (1.0 - sigma)
        /
        (1.0 - sigma)
    )


def evaluate_ppo_vectorized(
    model,
    n_episodes=2000,
    horizon=500,
    seed=10000
):

    rng = np.random.default_rng(seed)

    assets = np.ones(
        n_episodes,
        dtype=np.float64
    )

    income_states = np.zeros(
        n_episodes,
        dtype=int
    )

    total_utility = np.zeros(
        n_episodes,
        dtype=np.float64
    )

    discount = 1.0

    all_raw_actions = []

    for _ in range(horizon):

        incomes = np.where(
            income_states == 0,
            Y_HIGH,
            Y_LOW
        )

        obs = np.column_stack(
            [
                assets / 4.5,
                incomes
            ]
        ).astype(np.float32)

        raw_actions, _ = model.predict(
            obs,
            deterministic=True
        )

        raw_actions = np.asarray(
            raw_actions
        ).reshape(-1)

        z = action_to_saving_share(
            raw_actions
        )

        all_raw_actions.append(
            raw_actions.copy()
        )

        cash_on_hand = (
            R * assets + incomes
        )

        consumption = (
            (1.0 - z)
            * cash_on_hand
        )

        reward = utility_vectorized(
            consumption
        )

        total_utility += (
            discount * reward
        )

        assets = (
            z * cash_on_hand
        )

        p_low = np.where(
            income_states == 0,
            P_INCOME[0, 1],
            P_INCOME[1, 1]
        )

        income_states = (
            rng.random(n_episodes)
            < p_low
        ).astype(int)

        discount *= BETA

    mean = total_utility.mean()
    std = total_utility.std(ddof=1)
    se = std / np.sqrt(n_episodes)
    ci = 1.96 * se

    raw_flat = np.concatenate(
        all_raw_actions
    )

    return {
        "mean_utility": mean,
        "std_utility": std,
        "ci_lower": mean - ci,
        "ci_upper": mean + ci,
        "max_final_asset": assets.max(),
        "raw_action_mean": raw_flat.mean(),
        "raw_action_min": raw_flat.min(),
        "raw_action_max": raw_flat.max()
    }


def get_ppo_policy_on_grid(
    model,
    asset_grid
):

    policy = np.zeros(
        (len(asset_grid), 2)
    )

    raw_policy = np.zeros_like(
        policy
    )

    for income_state, income in [
        (0, Y_HIGH),
        (1, Y_LOW)
    ]:

        obs = np.column_stack(
            [
                asset_grid / 4.5,
                np.full(
                    len(asset_grid),
                    income
                )
            ]
        ).astype(np.float32)

        raw_actions, _ = model.predict(
            obs,
            deterministic=True
        )

        raw_actions = np.asarray(
            raw_actions
        ).reshape(-1)

        raw_policy[:, income_state] = (
            raw_actions
        )

        policy[:, income_state] = (
            action_to_saving_share(
                raw_actions
            )
        )

    return policy, raw_policy


print("Evaluation helpers loaded.")



## Heavy smoke-test cell

**계산부하: 중간~무거움**

- PPO 50,000 timesteps
- seed=42
- 이전 PPO와 hyperparameter는 동일
- **유일한 구조적 수정은 action 표현을 [0,0.95] → [-1,1]로 정규화한 것**

`Run all`에서는 학습되지 않습니다.


In [ ]:

RUN_HEAVY_TRAINING = False

if not RUN_HEAVY_TRAINING:

    print(
        "Fast setup complete.\n"
        "Change RUN_HEAVY_TRAINING = True and rerun THIS CELL ONLY."
    )

else:

    train_env = make_training_env()

    ppo_model = PPO(
        "MlpPolicy",
        train_env,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=64,
        gamma=BETA,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.0,
        verbose=1,
        seed=42,
        device="auto"
    )

    ppo_model.learn(
        total_timesteps=50_000
    )

    ppo_test = evaluate_ppo_vectorized(
        ppo_model,
        n_episodes=2000,
        horizon=500,
        seed=10000
    )

    ppo_policy, raw_policy = (
        get_ppo_policy_on_grid(
            ppo_model,
            asset_grid_fine
        )
    )

    ppo_error = np.abs(
        ppo_policy
        - policy_share_fine
    )

    print("\n=== NORMALIZED-ACTION PPO ===")
    print(
        "Mean utility:",
        ppo_test["mean_utility"]
    )
    print(
        "95% CI:",
        (
            ppo_test["ci_lower"],
            ppo_test["ci_upper"]
        )
    )
    print(
        "Policy MAE vs DP-Fine:",
        ppo_error.mean()
    )
    print(
        "Within 5pp:",
        np.mean(
            ppo_error <= 0.05
        )
    )
    print(
        "Within 10pp:",
        np.mean(
            ppo_error <= 0.10
        )
    )
    print(
        "Max final asset:",
        ppo_test["max_final_asset"]
    )

    print("\nRaw PPO action diagnostics [-1,1]")
    print(
        "Mean:",
        ppo_test["raw_action_mean"]
    )
    print(
        "Min:",
        ppo_test["raw_action_min"]
    )
    print(
        "Max:",
        ppo_test["raw_action_max"]
    )

    comparison = pd.DataFrame(
        [
            {
                "Model": "DP-Fine",
                "Mean Utility": -36.241647
            },
            {
                "Model": "Best DQN checkpoint",
                "Mean Utility": -36.492848
            },
            {
                "Model": "Old PPO [0,.95] action",
                "Mean Utility": -37.917680
            },
            {
                "Model": "PPO normalized action",
                "Mean Utility":
                    ppo_test["mean_utility"]
            }
        ]
    )

    display(comparison)

    plt.figure(figsize=(10, 5))

    plt.plot(
        asset_grid_fine,
        policy_share_fine[:, 0],
        label="DP-Fine: High"
    )

    plt.plot(
        asset_grid_fine,
        ppo_policy[:, 0],
        "--",
        label="PPO normalized: High"
    )

    plt.xlabel("Current Asset")
    plt.ylabel("Saving Share")
    plt.title("Normalized-Action PPO vs DP-Fine: High Income")
    plt.legend()
    plt.show()

    plt.figure(figsize=(10, 5))

    plt.plot(
        asset_grid_fine,
        policy_share_fine[:, 1],
        label="DP-Fine: Low"
    )

    plt.plot(
        asset_grid_fine,
        ppo_policy[:, 1],
        "--",
        label="PPO normalized: Low"
    )

    plt.xlabel("Current Asset")
    plt.ylabel("Saving Share")
    plt.title("Normalized-Action PPO vs DP-Fine: Low Income")
    plt.legend()
    plt.show()
